import libs

In [12]:
import os
from datetime import datetime
import pandas as pd
import urllib.request
import numpy as np

Частина 1 \
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;

In [13]:
# function for downloading and saving VHI data 
def download_VHI (province_id, folder="data"):
    # check if dir exist, and if not creates one
    if not os.path.exists(folder): 
        os.mkdir(folder)
        print(f"A new folder was created in current dir: {folder}")

    # check if there is file with data about this province and asks if u want to rewite it
    existing_files = [f for f in os.listdir(folder) if f.startswith(f"VHI_id_{province_id}_")]
    if existing_files:
        print(f"Data for province N{province_id} already exists: {existing_files[0]}")
        if input("Do you want to rewrite data about this region? (enter Y to rewrite):") == "Y":
            os.remove(os.path.join(folder, existing_files[0]))
        else:
            print("Downloading was canceled")
            return
    
    # prepairing some variables
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    cur_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"VHI_id_{province_id}_{cur_time}.csv"
    filepath = os.path.join(folder, filename)
    
    # Downloading and saving data to file
    try:
        print(f"Downloading data for province N{province_id}")
        with urllib.request.urlopen(url) as responce:
            data = responce.read().decode('utf-8')
        if len(data) < 100:
            print ("Too short responce, data was not saved.")
            return
        with open(filepath, 'w') as f:
            f.write(data)
        print(f"Data was saved to: {filepath}")
    except Exeption as e:
        print(f"Error when downloading data for province N{province_id}:{e}")

Run this to download data

In [4]:
for i in range (1,28):
    download_VHI(i)

Data was saved to: data\VHI_id_1_20260313_155438.csv
Data was saved to: data\VHI_id_2_20260313_155445.csv
Data was saved to: data\VHI_id_3_20260313_155446.csv
Data was saved to: data\VHI_id_4_20260313_155448.csv
Data was saved to: data\VHI_id_5_20260313_155449.csv
Data was saved to: data\VHI_id_6_20260313_155450.csv
Data was saved to: data\VHI_id_7_20260313_155452.csv
Data was saved to: data\VHI_id_8_20260313_155453.csv
Data was saved to: data\VHI_id_9_20260313_155454.csv
Data was saved to: data\VHI_id_10_20260313_155455.csv
Data was saved to: data\VHI_id_11_20260313_155456.csv
Data was saved to: data\VHI_id_12_20260313_155457.csv
Data was saved to: data\VHI_id_13_20260313_155458.csv
Data was saved to: data\VHI_id_14_20260313_155459.csv
Data was saved to: data\VHI_id_15_20260313_155500.csv
Data was saved to: data\VHI_id_16_20260313_155501.csv
Data was saved to: data\VHI_id_17_20260313_155502.csv
Data was saved to: data\VHI_id_18_20260313_155503.csv
Data was saved to: data\VHI_id_19_202

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області;
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька);

In [14]:
# NOAA index to Ukrainian index
ukr_sort_map = {23: 1, 24: 2, 5: 3, 6: 4, 26: 5, 22: 6, 25: 7, 7: 8, 11: 9, 12: 10, 13: 11, 14: 12, 15: 13, 16: 14, 17: 15, 18: 16, 20: 17, 21: 18, 8: 19, 9: 20, 10: 21, 1: 22, 2: 23, 3: 24, 4: 25, 19: 26, 27: 27}
# fuction to load data with pandas
def load_data (folder="data"):
    all_df = []
    for filename in os.listdir(folder):
        if filename.startswith("VHI_id_") and filename.endswith(".csv"): # read only files created by download_VHI to prevent errors
            filepath = os.path.join(folder,filename)
            try:
                # read first line of file to get region id and name
                file = open(filepath, "r")
                first_line = file.readline().split("Province= ")
                first_line = first_line[1].split(": ")
                region_id = ukr_sort_map.get(int(first_line[0]), int(first_line[0])) # reads+translates region id
                first_line = first_line[1].split(",")
                region_name = first_line[0]
    
                df = pd.read_csv(filepath, index_col=False, header=2, names=["year","week", "SMN","SMT","VCI","TCI", "VHI"], na_values=[-1.00]) # read data with pandas
                # data cleaning
                df = df[pd.to_numeric(df['year'], errors='coerce').notnull()] # deletes rows where year is not a number or is a null
                df['year'] = df['year'].astype(int)
                df['week'] = df['week'].astype(int)
                df = df.interpolate(method='linear') # fill Na wia interpolation
                df.insert(0,"region_id",region_id)
                df.insert(1,"region",region_name) 
                all_df.append(df)
                #print (df)
                #print('df done:',filename)
            except: print(f"wrong format, skipping {filename}")
    #print(all_df)
    # merge all df's into one
    if all_df:
        final_df = pd.concat(all_df, ignore_index=True)
        return(final_df)

In [15]:
df = load_data("data")
print ("region VHI data:")
display(df.head())

region VHI data:


,region_id,region,year,week,SMN,SMT,VCI,TCI,VHI
0,21,Khmel'nyts'kyy,1982,2,0.063,261.53,55.89,38.20,47.04
1,21,Khmel'nyts'kyy,1982,3,0.063,263.45,57.30,32.69,44.99
2,21,Khmel'nyts'kyy,1982,4,0.061,265.10,53.96,28.62,41.29
3,21,Khmel'nyts'kyy,1982,5,0.058,266.42,46.87,28.57,37.72
4,21,Khmel'nyts'kyy,1982,6,0.056,267.47,39.55,30.27,34.91


Series for:\
Реалізувати процедури для формування вибірок наступного виду:

1.VHI series for the region for the specified year;\
1.Ряд VHI для області за вказаний рік;

In [16]:
# IN - df, region id, year;   OUT - VHI series
def get_vhi_series(df, region_id, year):
    result = df[(df["region_id"] == region_id) & (df["year"] == year)]
    return (result['VHI'])

In [17]:
region_id= 5
year= 1990
vhi_series = get_vhi_series(df, region_id,year)
print(f"VHI series for region N{region_id} of {year} year")
display(vhi_series.head())

VHI series for region N5 of 1990 year


38410    39.21
38411    40.31
38412    42.10
38413    43.22
38414    43.36
Name: VHI, dtype: float64

2.VHI series for the specified range of years for the specified regions;\
2.Ряд VHI за вказаний діапазон років для вказаних областей;

In [18]:
# IN - df, list of ids, start year, end year;   OUT - VHI series
def get_vhi_range (df, region_ids, start_year, end_year):
    result = df[(df["region_id"].isin(region_ids)) & (df["year"] <= end_year) & (df["year"] >= start_year)]
    return (result['VHI'])

In [19]:
ids=[23,24]
start_year=1985
end_year=1990
print (f"VHI series for regions N{ids} between {start_year} and {end_year} years")
display(get_vhi_range(df,ids ,start_year, end_year))

VHI series for regions N[23, 24] between 1985 and 1990 years


42620    26.306667
42621    27.823333
42622    29.340000
42623    30.856667
42624    32.373333
           ...    
45162    32.040000
45163    30.930000
45164    28.720000
45165    28.870000
45166    31.090000
Name: VHI, Length: 624, dtype: float64

3.Search for extremes (min and max) for specified regions and years, average, median;\
3.Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;

In [20]:
# IN - df, list if ids, list of years;   OUT - dictionary with min, max, mean and median
def get_vhi_stats(df, region_ids, years): 
    result = df[(df["region_id"].isin(region_ids)) & (df["year"].isin(years))]
    stats = {'min': result['VHI'].min(), 'max':result['VHI'].max(), 'mean':result['VHI'].mean(), 'median':result['VHI'].median()}
    return (stats)

In [21]:
region_ids=[23,24]
years=[1985,1990]
stats=get_vhi_stats(df,region_ids, years)
print(f"VHI series for regions N{ids} for {years} years")
print(f"Min: {stats['min']},Max: {stats['max']}, Mean: {stats['mean']}, Median: {stats['median']}")

VHI series for regions N[23, 24] for [1985, 1990] years
Min: 26.2,Max: 65.8, Mean: 43.112067307692314, Median: 41.620000000000005
